In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import asyncio
from tqdm import tqdm
from btc_model.setting.setting import get_settings
from btc_model.core.util.file_util import FileUtil

In [3]:
import pandas as pd
import numpy as np
import os
import sys
import datetime
import pytz
from sqlalchemy import create_engine
import time
from copy import deepcopy
import traceback
import re
from urllib.parse import urlencode
import websocket
import json
import pickle
import datetime
import pandas as pd
import gc
import time
import hmac
import hashlib
import requests

/Users/Jason/work/source/03_ThorpAI/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
data_directory = FileUtil.get_project_dir(project_name='ThorpAI', sub_dir='data')
print(data_directory)

/Users/Jason/.ThorpAI/data


In [5]:
setting = get_settings('cex.binance.subaccount.jason')

In [6]:
setting

{'apikey': 'J4fdy5atEy5zMsGSaDef8VOvq3MlokmqbAFzeFj5hYg86UojL2PK32xoMm9ovQ8v',
 'secretkey': 'GwVuoLSOrXqZTnRbqSsUEvtpqRmVtxr4ExLWYo1KWtKkBaFMKlUsBlA4YtSQBxp2'}

In [7]:
BASE_URL = 'https://fapi.binance.com' # production base url
# BASE_URL = "https://testnet.binancefuture.com"  # testnet base url
BASE_URL_SPOT = 'https://api.binance.com'


KEY = setting['apikey']
SECRET = setting['secretkey']

In [8]:
""" ======  begin of functions, you don't need to touch ====== """


def hashing(query_string):
    return hmac.new(
        SECRET.encode("utf-8"), query_string.encode("utf-8"), hashlib.sha256
    ).hexdigest()


def get_timestamp():
    return int(time.time() * 1000)


def dispatch_request(http_method):
    session = requests.Session()
    session.headers.update(
        {"Content-Type": "application/json;charset=utf-8", "X-MBX-APIKEY": KEY}
    )
    return {
        "GET": session.get,
        "DELETE": session.delete,
        "PUT": session.put,
        "POST": session.post,
    }.get(http_method, "GET")


def send_signed_request(http_method, url_path, payload={}):
    query_string = urlencode(payload)
    query_string = query_string.replace("%27", "%22")

    if query_string:
        query_string = "{}&timestamp={}".format(query_string, get_timestamp())
    else:
        query_string = "timestamp={}".format(get_timestamp())

    url = (
        BASE_URL + url_path + "?" + query_string + "&signature=" + hashing(query_string)
    )
    print(url)
    params = {"url": url, "params": {}}
    response = dispatch_request(http_method)(**params)
    return response.json()


# used for sending public data request
def send_public_request(url_path, payload={}):
    query_string = urlencode(payload, True)
    url = BASE_URL + url_path
    if query_string:
        url = url + "?" + query_string
    print("{}".format(url))
    response = dispatch_request("GET")(url=url)
    return response.json()


def send_signed_request_spot(http_method, url_path, payload={}):
    query_string = urlencode(payload)
    query_string = query_string.replace("%27", "%22")

    if query_string:
        query_string = "{}&timestamp={}".format(query_string, get_timestamp())
    else:
        query_string = "timestamp={}".format(get_timestamp())

    url = (
        BASE_URL_SPOT + url_path + "?" + query_string + "&signature=" + hashing(query_string)
    )
    print(url)
    params = {"url": url, "params": {}}
    response = dispatch_request(http_method)(**params)
    return response.json()


""" ======  end of functions ====== """


' ======  end of functions ====== '

In [9]:
# Get Income History
startTime = datetime.datetime(2025,10,1,11,11,11)
endTime = datetime.datetime(2025,10,7,11,11,11)

Account_folder=data_directory.joinpath('binance_account')

for i in tqdm(range((endTime-startTime).days)):
    params = {"type": "FUTURES", 
              "startTime": int((startTime+datetime.timedelta(days=i)).timestamp()*1000), 
              "endTime": int((startTime+datetime.timedelta(days=i+1)).timestamp()*1000),
              "limit": "1000"}
    response = send_signed_request_spot("GET", "/sapi/v1/accountSnapshot", params)

    print('response:',len(response['snapshotVos']))
    
    if 'snapshotVos' in response and len(response['snapshotVos']) > 0:
        for j in range(len(response['snapshotVos'])):
            print(response['snapshotVos'][j])
            snapshot_data = response['snapshotVos'][j]['data']
            df_account = pd.DataFrame(snapshot_data['assets'])
            if len(df_account)==0:
                continue
            df_account['DATETIME']=pd.to_datetime(response['snapshotVos'][j]['updateTime'],unit='ms')
            df_account.to_csv(os.path.join(Account_folder,f'binance_account_{df_account["DATETIME"].max().strftime("%m_%d_%Y_%H_%M_%S")}.csv'))

    time.sleep(10)

account_result=[]
for p in os.listdir(Account_folder):
    df_account=pd.read_csv(os.path.join(Account_folder,p))
    value=df_account.loc[df_account['asset']=='USDT','marginBalance'].values[0]
    date=df_account['DATETIME'].max()
    account_result.append([date,value])
    print(date,value)

account_result=pd.DataFrame(account_result,columns=['Date','Value'])

  0%|          | 0/6 [00:00<?, ?it/s]

https://api.binance.com/sapi/v1/accountSnapshot?type=FUTURES&startTime=1759288271000&endTime=1759374671000&limit=1000&timestamp=1764081880254&signature=6500d4deb5986a53d2855382332303c055d086a9477c56b896caf57004f20ad1
response: 0


 17%|█▋        | 1/6 [00:10<00:52, 10.41s/it]

https://api.binance.com/sapi/v1/accountSnapshot?type=FUTURES&startTime=1759374671000&endTime=1759461071000&limit=1000&timestamp=1764081890663&signature=7fd35417eb54a6369f3f773db28d960c689477f019bb663743a3186ef05a3cfb
response: 0


 33%|███▎      | 2/6 [00:20<00:41, 10.31s/it]

...

https://api.binance.com/sapi/v1/accountSnapshot?type=FUTURES&startTime=1759633871000&endTime=1759720271000&limit=1000&timestamp=1764081921689&signature=4bcac105825a49b4ebb22267f630b00510d1fda5ac82023f1a463fcc1323e646
response: 0


 83%|████████▎ | 5/6 [00:51<00:10, 10.38s/it]

https://api.binance.com/sapi/v1/accountSnapshot?type=FUTURES&startTime=1759720271000&endTime=1759806671000&limit=1000&timestamp=1764081932142&signature=43ccf234213ac04ad3f5d5803bdd39140ef50d467bed6c2fa24a7c76667bd458
response: 0


100%|██████████| 6/6 [01:02<00:00, 10.35s/it]

2025-10-05 23:59:59 10200.51546461
2025-10-04 23:59:59 10134.28732644
2025-10-01 23:59:59 10028.2368981
2025-10-07 23:59:59 10400.1576681
2025-10-02 23:59:59 9945.24294436
2025-09-30 23:59:59 9986.75242989
2025-10-06 23:59:59 10238.02640905
2025-10-03 23:59:59 9990.95676025


In [10]:
account_result.sort_values(by='Date',inplace=True)

In [11]:
account_result

,Date,Value
5,2025-09-30 23:59:59,9986.752430
2,2025-10-01 23:59:59,10028.236898
4,2025-10-02 23:59:59,9945.242944
7,2025-10-03 23:59:59,9990.956760
1,2025-10-04 23:59:59,10134.287326
0,2025-10-05 23:59:59,10200.515465
6,2025-10-06 23:59:59,10238.026409
3,2025-10-07 23:59:59,10400.157668


In [81]:
# Get Income History
startTime = datetime.datetime(2025,10,7,0,0,0)
endTime = datetime.datetime(2025,10,7,23,59,59)


params = {"type": "FUTURES", 
            "startTime": int((startTime).timestamp()*1000), 
            "endTime": int((endTime).timestamp()*1000),
            "limit": "1000"}
response = send_signed_request_spot("GET", "/sapi/v1/accountSnapshot", params)

print('datetime:', pd.to_datetime(response['snapshotVos'][0]['updateTime'],unit='ms'))
print('assets:', response['snapshotVos'][0]['data']['assets'])


https://api.binance.com/sapi/v1/accountSnapshot?type=FUTURES&startTime=1759766400000&endTime=1759852799000&limit=1000&timestamp=1760244207319&signature=3bf3481c77fa4cece4307bb14e678f792ab8036d980e0ec76dcb5761a171f0f8


KeyError: 'snapshotVos'

In [56]:
pd.to_datetime(response['snapshotVos'][1]['updateTime'],unit='ms')

Timestamp('2025-10-07 23:59:59')

In [61]:
print(response['snapshotVos'][1]['data']['assets'])
print(pd.to_datetime(response['snapshotVos'][1]['updateTime'],unit='ms'))


[{'asset': 'USDT', 'marginBalance': '10139.65247753', 'walletBalance': '10110.02915918'}]
2025-10-07 23:59:59


In [62]:
response['snapshotVos'][1]['data']


{'assets': [{'asset': 'USDT',
   'marginBalance': '10139.65247753',
   'walletBalance': '10110.02915918'}],
 'position': [{'symbol': 'AAVEUSDT',
   'entryPrice': '287.048',
   'markPrice': '275.99058333',
   'positionAmt': '1',
   'unRealizedProfit': '-1.798'},
  {'symbol': 'ENSUSDT',
   'entryPrice': '0',
   'markPrice': '20.625',
   'positionAmt': '0',
   'unRealizedProfit': '0'},
  {'symbol': 'TONUSDT',
   'entryPrice': '2.842',
   'markPrice': '2.7521',
   'positionAmt': '-11.9',
   'unRealizedProfit': '-0.28917'},
  {'symbol': 'AUSDT',
   'entryPrice': '0.40643855',
   'markPrice': '0.3981',
   'positionAmt': '-3009',
   'unRealizedProfit': '31.71048835'}]}